# S1 equities — leverage (vol-target policy)

This notebook chooses a **holdable** `target_ann_vol` on unlevered weekly base returns. Arithmetic Sharpe is roughly invariant to constant leverage; CAGR, Calmar, drawdown, and CVaR are not. It is **not** a prop-firm pass-rate study and it does not re-run the 07 IS bakeoff.

**Run first.** This notebook does not fetch, score, or rebuild the book: it only reads the exported unlevered base parquet. Missing file → loud error naming the export notebook and path.

- Freeze stars from `04_backtest/s1_equities/notebooks/` 01–07, and have the IS predictions parquet that `08_oos_tearsheet.ipynb` already loads (`prediction_path_for_timing` → typically `03_models/s1_equities/model_artifacts/s1_linear_slim_ffill_is_predictions.parquet`). Desk `VT_TARGET_ANN_VOL_STAR` lives in `04_backtest/s1_equities/artifacts/s1_star_stack.json` (this notebook does not write it).
- Then run `04_backtest/s1_equities/notebooks/08_oos_tearsheet.ipynb` so both exist:
  - `01_data/data_files/s1_equities/s1_period_returns_base.parquet` — unlevered weekly `ret` (this notebook)
  - `01_data/data_files/s1_equities/s1_period_returns.parquet` — net post-VT (later MC / prop-firm)

Half-Kelly is betting about half the theoretically growth-optimal fraction so noisy estimates of edge do not blow the account; CAGR is the constant yearly rate that turns $1 into ending wealth; Calmar is that CAGR divided by the worst peak-to-trough loss; and CVaR is the average outcome in the worst tail (here 5%).

## 0. Imports & Config


In [ ]:
import os
import sys

import pandas as pd
from IPython.display import display

cur = os.path.abspath(os.getcwd())
ROOT = cur
for _ in range(12):
    if os.path.isfile(os.path.join(cur, "pyproject.toml")) and os.path.isdir(
        os.path.join(cur, "06_risk")
    ):
        ROOT = cur
        break
    parent = os.path.dirname(cur)
    if parent == cur:
        break
    cur = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from risk.analytics.leverage.apply import s1_frozen_cfg
from risk.analytics.leverage.loaders import load_s1_period_returns_base
from risk.analytics.leverage.plots import surface_cagr_figure, surface_calmar_figure, surface_dd_figure
from risk.analytics.leverage.report import run_leverage_policy
from risk.analytics.monte_carlo.loaders import find_repo_root

ROOT = find_repo_root(ROOT)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("ROOT", ROOT)

SLEEVE = "s1"
BAR = "W"
PERIODS_PER_YEAR = 52
VT_STAR = "vt_bayes_0.9_10_q0.75_db0.05"
DEFAULT_IS_END = "2021-12-31"
DEFAULT_TARGETS = [0.06, 0.08, 0.10, 0.12, 0.15, 0.18]
DEFAULT_DD_CAP = 0.25
PICK = "calmar"
CFG = s1_frozen_cfg(VT_STAR, periods_per_year=PERIODS_PER_YEAR)
print("VT_STAR", VT_STAR)
print("desk target lives in s1_star_stack.json (this notebook does not write it)")


## 1. Data Loading


In [ ]:
BASE = load_s1_period_returns_base(ROOT)
print("base bars", len(BASE), BASE.index.min().date(), BASE.index.max().date())
print(BASE.tail())


## 2. Vol-target surface

Frozen VT family; only `target_ann_vol` changes. IS and OOS are reported separately. Half-Kelly is a **ceiling**, not an objective.


In [ ]:
PACK = {}

def run(is_end, max_oos_dd):
    pack = run_leverage_policy(
        BASE,
        CFG,
        targets=list(DEFAULT_TARGETS),
        is_end=is_end,
        periods_per_year=PERIODS_PER_YEAR,
        max_oos_dd=float(max_oos_dd),
        pick=PICK,
        sleeve=SLEEVE,
        vt_star=VT_STAR,
    )
    PACK.clear()
    PACK.update(pack)
    print("This overlay is live VT on **base** returns, not r'=k r on the sealed net parquet.")
    print("half-Kelly vol ceiling (not an objective)", pack["half_kelly_vol"])
    display(pack["surface"])
    display(pd.Series(pack["decision"], name="policy").to_frame("value"))
    display(surface_cagr_figure(pack["surface"]))
    display(surface_calmar_figure(pack["surface"]))
    display(surface_dd_figure(pack["surface"]))
    return pack

pack = run(DEFAULT_IS_END, DEFAULT_DD_CAP)
try:
    import ipywidgets as w
    ui = w.interactive(
        run,
        is_end=w.Text(value=str(DEFAULT_IS_END), description="IS end"),
        max_oos_dd=w.FloatSlider(min=0.05, max=0.50, value=DEFAULT_DD_CAP, step=0.05, description="DD veto"),
    )
    display(ui)
except Exception as exc:
    print("ipywidgets unavailable (%s); default run already executed" % exc)


## 3. Policy pick


In [ ]:
if not PACK:
    raise RuntimeError("run() did not populate PACK")
dec = PACK["decision"]
print("recommended target_ann_vol", dec.get("target_ann_vol"), "reason", dec.get("reason"))
print("survivors", dec.get("n_survivors"), "half-Kelly ceiling", PACK["half_kelly_vol"])
print("desk pick (not written)", dec.get("target_ann_vol"))
display(pd.Series(dec, name="policy").to_frame("value"))


## 4. Evaluation

- OOS Sharpe should be roughly flat across the vol grid (constant-$k$ invariance of arithmetic Sharpe).
- Pick is max OOS **Calmar** among targets that pass the drawdown veto and sit at or below half-Kelly vol.
- Label: live VT overlay on **base** returns. Stops/costs scale with gross in the full runner; this is the same approximation 07 already uses.
- Next: `02_ev_vs_spy.ipynb` (geometry on sealed net) and `prop_firm/` (FTMO-capped $k$).


In [ ]:
print("sleeve", SLEEVE, "pick", PICK)
print("half-Kelly is a ceiling, not an objective")
if PACK.get("decision"):
    print(PACK["decision"])
